# Meeting Recording Summarizer

Upload a meeting recording and click **Meeting summary** to transcribe the audio and stream a structured recap.

In [ ]:
import os
from typing import Generator
import textwrap

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise RuntimeError("OPENAI_API_KEY is not set. Please add it to your environment before running this notebook.")

client = OpenAI(api_key=openai_api_key)

TRANSCRIPTION_MODEL = "whisper-1"
SUMMARY_MODEL = "gpt-4o-mini"


In [ ]:
def stream_summary_markdown(transcript_text: str) -> Generator[str, None, None]:
    system_prompt = textwrap.dedent(
        """
        You are a helpful assistant who produces concise meeting minutes.
        Respond in GitHub-flavored Markdown.
        Always include the following sections in this exact order with the same headings:
        ## Summary
        ## Attendees and Date
        ## Decisions Made
        ## Actions with Owners
        Populate each section using the transcript. If details are missing, write 'Not discussed'.
        Infer attendees and meeting date from the transcript when possible; if the transcript omits them, state 'Not discussed'.
        List action items as bullet points that name each owner.
        """
    ).strip()

    user_prompt = textwrap.dedent(
        f"""
        Meeting transcript:
        {transcript_text}
        """
    ).strip()

    accumulated_markdown = ""

    with client.responses.stream(
        model=SUMMARY_MODEL,
        input=[
            {"role": "system", "content": [{"type": "input_text", "text": system_prompt}]},
            {"role": "user", "content": [{"type": "input_text", "text": user_prompt}]},
        ],
        max_output_tokens=900,
    ) as stream:
        for event in stream:
            if event.type == "response.output_text.delta":
                chunk = event.delta or ""
                if chunk:
                    accumulated_markdown += chunk
                    yield accumulated_markdown
            elif event.type == "response.error" and hasattr(event, "error"):
                message = getattr(event.error, "message", None) or "Unexpected error from summary model."
                raise RuntimeError(message)
        stream.get_final_response()


def summarize_meeting(meeting_audio_path: str):
    if not meeting_audio_path:
        yield "⚠️ Please upload a meeting recording before requesting a summary."
        return

    yield "_Transcribing audio with Whisper..._"

    try:
        with open(meeting_audio_path, "rb") as audio_file:
            transcription_result = client.audio.transcriptions.create(
                model=TRANSCRIPTION_MODEL,
                file=audio_file,
                response_format="text",
            )
    except Exception as exc:
        yield f"❌ Transcription failed: {exc}"
        return

    if isinstance(transcription_result, str):
        transcript_text = transcription_result.strip()
    else:
        transcript_text = getattr(transcription_result, "text", "").strip()

    if not transcript_text:
        yield "⚠️ The transcription is empty. Please check the audio quality or try another recording."
        return

    yield "_Generating meeting summary..._"

    try:
        for chunk in stream_summary_markdown(transcript_text):
            yield chunk
    except Exception as exc:
        yield f"❌ Summary generation failed: {exc}"


def on_audio_upload(_: str):
    return ""


def clear_inputs():
    return gr.Audio.update(value=None), ""


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## Meeting summary generator")
    audio_input = gr.Audio(
        label="Meeting recording",
        sources=["upload"],
        type="filepath",
    )
    with gr.Row():
        summarize_button = gr.Button("Meeting summary")
        clear_button = gr.Button("Clear", variant="secondary")
    summary_output = gr.Markdown()

    summarize_button.click(
        fn=summarize_meeting,
        inputs=[audio_input],
        outputs=summary_output,
    )

    audio_input.upload(
        fn=on_audio_upload,
        inputs=[audio_input],
        outputs=summary_output,
    )

    clear_button.click(
        fn=clear_inputs,
        inputs=None,
        outputs=[audio_input, summary_output],
    )

demo.queue()
demo.launch(debug=True)
